## Cell 1 — Config: MongoDB connection, target columns, ADLS path

**TODO before running:**
- Confirm `SELECTED_COLUMNS` against the source-of-truth column list (pasted text, not photos — several fields here are still placeholders pending confirmation).
- Confirm `INCREMENTAL_COLUMN` (the "service request number" field used as the watermark).
- Confirm `MONGO_COLLECTION` name inside the `servicerequest` database.
- Store the Mongo password in a Databricks secret scope — never hardcode it.
- Make `ca.cert` available to the cluster (e.g. via a Unity Catalog Volume or DBFS path) and point `CA_FILE` at it.

In [ ]:
import certifi
from datetime import datetime, timezone
from pymongo import MongoClient
from pyspark.sql import functions as F

# ── MongoDB connection (Xpress Form DB) ──────────────────────────────────────
MONGO_USER        = "xpreaduser"
MONGO_PASSWORD    = dbutils.secrets.get(scope="<SECRET_SCOPE>", key="xpress-mongo-password")
MONGO_HOSTS        = (
    "HBXPRESSFRMDRDB1.hbctxdom.com:28181,"
    "HBXPRBPRDQH01.hbctxdom.com:28181,"
    "HBXPRESSFRMPRDDB1.hbctxdom.com:28181"
)
MONGO_DB           = "servicerequest"
MONGO_COLLECTION   = "service_request"  # TODO confirm exact collection name
REPLICA_SET        = "xpressform"
CA_FILE            = "/Volumes/<catalog>/<schema>/<volume>/ca.cert"  # TODO point at the actual cert location

MONGO_URI = (
    f"mongodb://{MONGO_USER}:{MONGO_PASSWORD}@{MONGO_HOSTS}/"
    f"?replicaSet={REPLICA_SET}&tls=true&authSource={MONGO_DB}"
)

# ── Columns required in the final output (from Xpress_Forms_mapping_sheet) ──
# TODO: replace with the verified, pasted column list (this is the best-effort
# OCR read from photos and must be cross-checked before this notebook is trusted).
SELECTED_COLUMNS = [
    # paste verified column list here
]

# ── Incremental watermark ────────────────────────────────────────────────────
INCREMENTAL_COLUMN = "userReferenceNumber"  # TODO confirm this is the "service request number" field

# ── ADLS target ───────────────────────────────────────────────────────────────
ADLS_PATH = (
    "abfss://raw@ddiprodvyapaaradlsstd.dfs.core.windows.net/"
    "raw_data/business_banking/merchant/vyapaar/elastic/wow_journey_mongodb_service_request/"
)


## Cell 2 — Determine watermark

Reads the max value of `INCREMENTAL_COLUMN` already landed in ADLS so this run only
pulls new/changed service requests. If no data exists yet at `ADLS_PATH`, this is a
full first load.

In [ ]:
try:
    existing_df  = spark.read.parquet(ADLS_PATH)
    last_value   = existing_df.agg(F.max(INCREMENTAL_COLUMN)).first()[0]
    print(f"Resuming after {INCREMENTAL_COLUMN} = {last_value!r}")
except Exception as e:
    last_value = None
    print(f"No existing data at {ADLS_PATH} — running a full load. ({e})")


## Cell 3 — Fetch from MongoDB in batches & build DataFrame

Pulls only `SELECTED_COLUMNS`, filtered to documents newer than the watermark, in
fixed-size batches (mirrors the windowed approach in `windowed_data_loader.ipynb` —
avoids loading the whole collection into driver memory at once).

In [ ]:
BATCH_SIZE = 20_000

client     = MongoClient(MONGO_URI, tlsCAFile=CA_FILE)
collection = client[MONGO_DB][MONGO_COLLECTION]

mongo_filter = {INCREMENTAL_COLUMN: {"$gt": last_value}} if last_value is not None else {}
projection   = {col: 1 for col in SELECTED_COLUMNS}
projection["_id"] = 0

cursor = collection.find(mongo_filter, projection).batch_size(BATCH_SIZE)

df = None
batch = []

def flush(batch, df):
    if not batch:
        return df
    batch_df = spark.createDataFrame(batch)
    return batch_df if df is None else df.union(batch_df)

for i, doc in enumerate(cursor, start=1):
    batch.append(doc)
    if len(batch) >= BATCH_SIZE:
        print(f"  flushing batch ending at record {i:,}")
        df = flush(batch, df)
        batch = []

df = flush(batch, df)
client.close()

if df is None:
    print("No new records since last watermark — nothing to write.")
else:
    print(f"total new records: {df.count():,}, columns: {len(df.columns)}")


## Cell 4 — Write to ADLS

In [ ]:
if df is not None:
    (
        df.withColumn("ingestion_date", F.current_date())
          .write
          .mode("append")
          .partitionBy("ingestion_date")
          .parquet(ADLS_PATH)
    )
    print(f"wrote {df.count():,} records to {ADLS_PATH}")
